In [122]:
import pandas as pd

In [131]:
df_dim_region   = pd.read_parquet('../../data/silver/dim_region.parquet')
df_dim_product  = pd.read_parquet('../../data/silver/dim_product.parquet')
df_dim_calendar = pd.read_parquet('../../data/silver/dim_calendar.parquet')
df_dim_supplier = pd.read_parquet('../../data/silver/dim_supplier.parquet')
df_fact_sales   = pd.read_parquet('../../data/silver/fact_sales.parquet')

## Active combinations

Only (supplier, product, region) combos that actually had sales are kept.
This naturally excludes empty time series without any extra filtering step.

In [132]:
# Aggregate fact_sales to ISO week level — fixes pharma-week start inconsistencies
# (e.g. Nov-01 Tue and Nov-06 Sun both belong to ISO week Oct-31 and are summed)
weekly_sales = (
    df_fact_sales
    .merge(df_dim_calendar[['date_id', 'week_date']], on='date_id', how='left')
    .groupby(['week_date', 'supplier_id', 'region_id', 'product_id'], as_index=False)
    ['units_sold'].sum()
)

#get index of empty timeseries
empty_timeseries = weekly_sales.groupby(['supplier_id','region_id','product_id']).agg({'units_sold':'sum'}).reset_index().query('units_sold == 0')[['supplier_id','region_id','product_id']]

# Remove empty timeseries from active combos
weekly_sales = (
    weekly_sales.merge(empty_timeseries, on=['supplier_id','region_id','product_id'], how='left', indicator=True)
    .query('_merge == "left_only"')
    .drop(columns=['_merge'])
)

# Active combos derived from weekly_sales — guarantees at least one non-zero week per combo
active_combos = (
    weekly_sales[['supplier_id', 'region_id', 'product_id']]
    .drop_duplicates()
)

print(f'Active combinations: {len(active_combos):,}')

Active combinations: 2,841


## Weekly calendar

The forecast target is weekly — no need for daily granularity here.
One row per week keeps the cross join manageable.

In [133]:
weekly_calendar = (
    df_dim_calendar
    .drop_duplicates(subset='week_date')
    [['week_date', 'year', 'semester', 'quarter', 'month']]
    .reset_index(drop=True)
)

print(f'Weeks in calendar: {len(weekly_calendar)}')

Weeks in calendar: 104


## Build main dataset

Cross join: active_combos × weeks → expected size ≈ active_combos × 124 weeks.
Left join with fact_sales fills actual sales; missing weeks become NaN → 0.

In [134]:
active_combos = (
    df_fact_sales[['supplier_id', 'region_id', 'product_id']]
    .drop_duplicates()
)

In [135]:
dataset

,week_date,year,semester,quarter,month,units_sold,region_name,product_attribute_1,product_attribute_2,product_attribute_3,product_name,supplier_name
0,2022-10-31,2022,2,4,11,2.0,Sul,A,A,A03,Product 97,Supplier 1
1,2022-11-07,2022,2,4,11,0.0,Sul,A,A,A03,Product 97,Supplier 1
2,2022-11-14,2022,2,4,11,0.0,Sul,A,A,A03,Product 97,Supplier 1
3,2022-11-21,2022,2,4,11,0.0,Sul,A,A,A03,Product 97,Supplier 1
4,2022-11-28,2022,2,4,11,0.0,Sul,A,A,A03,Product 97,Supplier 1
...,...,...,...,...,...,...,...,...,...,...,...,...
295875,2024-09-23,2024,2,3,9,0.0,Norte,C,B,B04,Product 95,Supplier 24
295876,2024-09-30,2024,2,3,9,0.0,Norte,C,B,B04,Product 95,Supplier 24
295877,2024-10-07,2024,2,4,10,0.0,Norte,C,B,B04,Product 95,Supplier 24
295878,2024-10-14,2024,2,4,10,0.0,Norte,C,B,B04,Product 95,Supplier 24


In [139]:
dataset = (
    active_combos
    .merge(weekly_calendar, how='cross')
    .merge(
        weekly_sales,
        on=['week_date', 'supplier_id', 'region_id', 'product_id'],
        how='left'
    )
    .merge(df_dim_region,   on='region_id',   how='left')
    .merge(df_dim_product,  on='product_id',  how='left')
    .merge(df_dim_supplier, on='supplier_id', how='left')
    .drop(columns=['region_id', 'product_id', 'supplier_id'])
)

dataset['units_sold'] = dataset['units_sold'].fillna(0)

del active_combos, weekly_calendar, weekly_sales

print(f'Dataset shape: {dataset.shape}')

Dataset shape: (295880, 12)


In [140]:
dataset.head()

,week_date,year,semester,quarter,month,units_sold,region_name,product_attribute_1,product_attribute_2,product_attribute_3,product_name,supplier_name
0,2022-10-31,2022,2,4,11,0.0,Norte,B,C,B24,Product 233,Supplier 6
1,2022-11-07,2022,2,4,11,0.0,Norte,B,C,B24,Product 233,Supplier 6
2,2022-11-14,2022,2,4,11,0.0,Norte,B,C,B24,Product 233,Supplier 6
3,2022-11-21,2022,2,4,11,0.0,Norte,B,C,B24,Product 233,Supplier 6
4,2022-11-28,2022,2,4,11,0.0,Norte,B,C,B24,Product 233,Supplier 6


## Memory optimization

Downcast numerics and convert low-cardinality strings to `category`.

In [141]:
dataset['units_sold'] = dataset['units_sold'].astype('float32')

for col in ['year', 'month', 'quarter', 'semester']:
    dataset[col] = dataset[col].astype('int16')

for col in ['region_name', 'product_name', 'supplier_name']:
    dataset[col] = dataset[col].astype('category')

print(dataset.dtypes)
print(f'\nMemory usage: {dataset.memory_usage(deep=True).sum() / 1e6:.1f} MB')

week_date              datetime64[ns]
year                            int16
semester                        int16
quarter                         int16
month                           int16
units_sold                    float32
region_name                  category
product_attribute_1            object
product_attribute_2            object
product_attribute_3            object
product_name                 category
supplier_name                category
dtype: object

Memory usage: 59.2 MB


In [143]:
dataset.head()

,week_date,year,semester,quarter,month,units_sold,region_name,product_attribute_1,product_attribute_2,product_attribute_3,product_name,supplier_name
0,2022-10-31,2022,2,4,11,0.0,Norte,B,C,B24,Product 233,Supplier 6
1,2022-11-07,2022,2,4,11,0.0,Norte,B,C,B24,Product 233,Supplier 6
2,2022-11-14,2022,2,4,11,0.0,Norte,B,C,B24,Product 233,Supplier 6
3,2022-11-21,2022,2,4,11,0.0,Norte,B,C,B24,Product 233,Supplier 6
4,2022-11-28,2022,2,4,11,0.0,Norte,B,C,B24,Product 233,Supplier 6


In [146]:
dataset.to_parquet('../../data/gold/dataset_vendas.parquet', index=False)
print('Saved.')

Saved.


In [145]:
dataset.groupby('week_date')['units_sold'].sum().head(50)

week_date
2022-10-31    114581.0
2022-11-07     64134.0
2022-11-14     68566.0
2022-11-21     39839.0
2022-11-28     88060.0
2022-12-05     75696.0
2022-12-12     62304.0
2022-12-19     54763.0
2022-12-26     54097.0
2023-01-02     56952.0
2023-01-09     56733.0
2023-01-16     62554.0
2023-01-23     30822.0
2023-01-30    106439.0
2023-02-06     65540.0
2023-02-13     55488.0
2023-02-20     33556.0
2023-02-27    115152.0
2023-03-06     71110.0
2023-03-13     77626.0
2023-03-20     79519.0
2023-03-27     67283.0
2023-04-03     69506.0
2023-04-10     61392.0
2023-04-17     83935.0
2023-04-24      2601.0
2023-05-01    141856.0
2023-05-08     78464.0
2023-05-15     77168.0
2023-05-22     49890.0
2023-05-29    112537.0
2023-06-05     80795.0
2023-06-12     82739.0
2023-06-19     86459.0
2023-06-26     99987.0
2023-07-03     53338.0
2023-07-10     46171.0
2023-07-17     51006.0
2023-07-24    191231.0
2023-07-31     77255.0
2023-08-07     42431.0
2023-08-14     34465.0
2023-08-21     88313.0
2